# Rolling and Expanding Windows

Window functions compute statistics over subsets of data — essential for
smoothing noisy signals, estimating volatility, and engineering features for
forecasting models. Pandas provides two main flavours:

- **Rolling** windows have a fixed size and slide forward one observation at a time.
- **Expanding** windows grow with each new observation, accumulating all data seen so far.

This notebook covers both, culminating in a practical Bollinger Bands example
that combines rolling mean and rolling standard deviation.

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Plotting defaults
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

In [ ]:
# --- Airline Passengers (monthly, 1949-1960) ---
airline = pd.read_csv(
    DATA_DIR / "airline_passengers.csv",
    index_col="Month",
    parse_dates=True,
)
airline.index.freq = "MS"  # month-start frequency
airline.columns = ["Passengers"]

# --- Alcohol Sales (monthly, 1992-2019) ---
alcohol = pd.read_csv(
    DATA_DIR / "Alcohol_Sales.csv",
    index_col="DATE",
    parse_dates=True,
)
alcohol.index.freq = "MS"
alcohol.columns = ["Sales"]

print("Airline shape:", airline.shape)
print("Alcohol shape:", alcohol.shape)
airline.head()

## Simple Moving Average (SMA) with `rolling()`

The **Simple Moving Average** replaces each observation with the mean of the
preceding *k* observations (the *window*):

$$
SMA_t = \frac{1}{k} \sum_{i=0}^{k-1} y_{t-i}
$$

In pandas the call is `df.rolling(window=k).mean()`.  By default the window is
**trailing** (the current observation is the rightmost point) and the first
`k - 1` values are `NaN` because there are not enough observations yet.

In [ ]:
# 12-month simple moving average for airline passengers
airline["SMA_12"] = airline["Passengers"].rolling(window=12).mean()

airline[["Passengers", "SMA_12"]].head(15)

In [ ]:
# Visualize: original data + 12-month SMA overlaid
fig, ax = plt.subplots()
ax.plot(airline.index, airline["Passengers"], label="Original", alpha=0.7)
ax.plot(airline.index, airline["SMA_12"], label="12-month SMA", linewidth=2.5)
ax.set_title("Airline Passengers — 12-Month Simple Moving Average")
ax.set_xlabel("Date")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
plt.tight_layout()
plt.show()

The SMA strips away seasonal fluctuations and reveals the underlying **trend**.

### `min_periods` and `center`

- `min_periods`: minimum number of non-NaN observations required to produce a
  value.  Default equals `window`, but setting it lower lets the SMA start
  earlier (with fewer data points).
- `center=True`: places the window label at the centre rather than the right
  edge.  Useful for symmetric smoothing (but introduces future information, so
  do **not** use for forecasting).

In [ ]:
# Compare trailing vs centred SMA, and the effect of min_periods
passengers = airline["Passengers"]

sma_trailing = passengers.rolling(window=12).mean()
sma_centered = passengers.rolling(window=12, center=True).mean()
sma_min3 = passengers.rolling(window=12, min_periods=3).mean()

fig, ax = plt.subplots()
ax.plot(passengers, label="Original", alpha=0.5)
ax.plot(sma_trailing, label="Trailing (default)")
ax.plot(sma_centered, label="Centered", linestyle="--")
ax.plot(sma_min3, label="min_periods=3", linestyle=":")
ax.set_title("Effect of center and min_periods on Rolling Mean")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
plt.tight_layout()
plt.show()

## Rolling Standard Deviation

Rolling standard deviation measures **volatility** — how spread out the data is
within each window.  If the rolling std is increasing over time, the series
exhibits **heteroscedasticity** (non-constant variance), which often points to
multiplicative seasonality.

In [ ]:
# 12-month rolling standard deviation
airline["Rolling_Std_12"] = passengers.rolling(window=12).std()

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(passengers, label="Passengers")
axes[0].set_title("Airline Passengers")
axes[0].set_ylabel("Thousands")
axes[0].legend()

axes[1].plot(airline["Rolling_Std_12"], color="tab:orange", label="12-month Rolling Std")
axes[1].set_title("Rolling Standard Deviation (window=12)")
axes[1].set_ylabel("Std Dev")
axes[1].set_xlabel("Date")
axes[1].legend()

plt.tight_layout()
plt.show()

print("The increasing rolling std confirms that variance grows with the level")
print("of the series — a hallmark of multiplicative seasonality.")

## Other Rolling Aggregations

The `.rolling()` object supports many built-in aggregations and also accepts
arbitrary functions via `.apply()`.

In [ ]:
window = 12

rolling_stats = pd.DataFrame({
    "Original": passengers,
    "Min":    passengers.rolling(window).min(),
    "Max":    passengers.rolling(window).max(),
    "Median": passengers.rolling(window).median(),
    "Sum":    passengers.rolling(window).sum(),
    "Var":    passengers.rolling(window).var(),
})

rolling_stats.tail(10)

In [ ]:
# Rolling correlation between airline passengers and its own 6-month lag
lagged = passengers.shift(6)
rolling_corr = passengers.rolling(window=24).corr(lagged)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(rolling_corr)
ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
ax.set_title("24-Month Rolling Correlation: Passengers vs 6-Month Lag")
ax.set_ylabel("Correlation")
plt.tight_layout()
plt.show()

In [ ]:
# Custom rolling function — rolling skewness
from scipy.stats import skew

rolling_skew = passengers.rolling(window=24).apply(skew, raw=True)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(rolling_skew, color="tab:green")
ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
ax.set_title("24-Month Rolling Skewness (custom function via .apply())")
ax.set_ylabel("Skewness")
plt.tight_layout()
plt.show()

## Time-Based Windows

Instead of specifying the window as an integer (number of rows), you can pass a
**time offset string** like `'90D'` (90 calendar days).  This is especially
useful for **irregularly spaced** data where each row does not represent the
same time interval.

| Syntax | Meaning |
|---|---|
| `rolling(90)` | 90 rows regardless of dates |
| `rolling('90D')` | 90 calendar days — adapts to gaps |

In [ ]:
# Create an irregularly spaced sample to illustrate the difference
np.random.seed(42)
irregular_dates = pd.to_datetime(
    sorted(np.random.choice(pd.date_range("2020-01-01", "2021-12-31"), size=80, replace=False))
)
irregular = pd.Series(
    np.cumsum(np.random.randn(len(irregular_dates))),
    index=irregular_dates,
    name="value",
)

# Integer window vs time-based window
sma_rows = irregular.rolling(30).mean()          # 30 observations
sma_days = irregular.rolling("90D").mean()        # 90 calendar days

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(irregular, label="Irregular data", alpha=0.5, marker="o", markersize=3)
ax.plot(sma_rows, label="rolling(30) — 30 rows")
ax.plot(sma_days, label="rolling('90D') — 90 calendar days", linestyle="--")
ax.set_title("Integer Window vs Time-Based Window on Irregular Data")
ax.legend()
plt.tight_layout()
plt.show()

## Expanding Windows

An **expanding** window includes all observations from the beginning of the
series up to the current point.  The window grows by one observation at each
step — there is no fixed size.

Common use cases:
- **Cumulative statistics**: running mean, running std
- **Running totals**: `expanding().sum()`
- **Benchmark comparisons**: compare each observation to the cumulative average

In [ ]:
# Expanding (cumulative) mean vs 12-month rolling mean
expanding_mean = passengers.expanding(min_periods=1).mean()
rolling_mean = passengers.rolling(window=12).mean()

fig, ax = plt.subplots()
ax.plot(passengers, label="Original", alpha=0.4)
ax.plot(rolling_mean, label="Rolling Mean (12)", linewidth=2)
ax.plot(expanding_mean, label="Expanding Mean", linewidth=2, linestyle="--")
ax.axhline(passengers.mean(), color="grey", linestyle=":", label="Global Mean")
ax.set_title("Rolling Mean vs Expanding Mean — Airline Passengers")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
plt.tight_layout()
plt.show()

print("The expanding mean converges toward the global mean as more data is included.")

In [ ]:
# Expanding standard deviation — grows then stabilises
expanding_std = passengers.expanding(min_periods=2).std()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(expanding_std, color="tab:red", label="Expanding Std")
ax.set_title("Expanding Standard Deviation — Airline Passengers")
ax.set_ylabel("Std Dev")
ax.legend()
plt.tight_layout()
plt.show()

## Bollinger Bands Example

**Bollinger Bands** are a practical application that combines a rolling mean
with a rolling standard deviation to create an envelope around the data:

- **Middle band** = SMA(*k*)
- **Upper band** = SMA(*k*) + 2 $\times$ rolling std(*k*)
- **Lower band** = SMA(*k*) − 2 $\times$ rolling std(*k*)

Originally developed for stock analysis, Bollinger Bands are also useful for
**anomaly detection** — any observation outside the bands may warrant
investigation.

In [ ]:
# Bollinger Bands on Alcohol Sales (12-month window)
sales = alcohol["Sales"]

sma = sales.rolling(12).mean()
std = sales.rolling(12).std()
upper = sma + 2 * std
lower = sma - 2 * std

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(sales, label="Alcohol Sales", alpha=0.6)
ax.plot(sma, label="12-Month SMA", color="tab:orange", linewidth=2)
ax.plot(upper, label="Upper Band (+2 std)", color="tab:red", linestyle="--", linewidth=1)
ax.plot(lower, label="Lower Band (-2 std)", color="tab:green", linestyle="--", linewidth=1)
ax.fill_between(sales.index, upper, lower, alpha=0.12, color="tab:blue")
ax.set_title("Bollinger Bands — Alcohol Sales (window=12)")
ax.set_xlabel("Date")
ax.set_ylabel("Sales (millions of dollars)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Identify observations outside the bands
outside = sales[(sales > upper) | (sales < lower)].dropna()
print(f"Observations outside Bollinger Bands: {len(outside)} out of {len(sales)}")
outside.head(10)

## Summary

| | **Rolling** | **Expanding** |
|---|---|---|
| **Window size** | Fixed (*k* observations) | Grows from 1 to *n* |
| **Key method** | `df.rolling(window)` | `df.expanding(min_periods)` |
| **Behaviour** | Slides forward, drops oldest point | Never drops — accumulates all data |
| **Typical use** | Smoothing, volatility, Bollinger Bands | Cumulative stats, running totals |
| **Key parameters** | `window`, `min_periods`, `center` | `min_periods` |
| **Time-based window** | `rolling('90D')` for irregular data | N/A (always uses all past data) |

In [ ]:
# Clean up helper columns added during exploration
airline.drop(columns=["SMA_12", "Rolling_Std_12"], inplace=True, errors="ignore")